# CPA Screening: end-to-end runner

This notebook reproduces every result in the README on a Colab Pro GPU runtime, top-to-bottom in ~25 min on Blackwell / ~45 min on T4. It's the canonical entry point: clone the repo, run all cells, download `results.zip`.

This version is the **v2 pipeline** (see the README's "v2: changes I made after looking at the v1 outputs" section). Key differences from the original Phase 1-3 build:

- **Concentration-aware toxicity**: every (compound, concentration) measurement from Higgins Dec 2025 is its own training row; toxicity head takes concentration as an explicit input. RF baseline toxicity Spearman jumped from 0.347 to 0.527 in CPU smoke tests just from this change.
- **Tighter CPA filter**: element whitelist {C, H, N, O, S}, MW [30, 350], logP < 1.5, ring count ≤ 2, no azo / multi-sulfonate. Rejects food dyes, organomercurials, halogenated phenolics, and inorganic acids that snuck through v1.
- **Tox21 aux head activated**: ChemBERTa now trains with a 12-class BCE auxiliary loss on 7,800 broader-toxicity compounds, weighted at 0.1 to regularize the encoder without dominating the CPA gradient. Loader is a direct CSV download; no DeepChem dep.

What it produces:
1. **Dataset audit**: counts, label ranges, PubChem hit rate (`data/processed/audit.json`)
2. **RF baseline (v2)**: single-seed and 5-seed cluster ensemble + LOO permeability, with concentration as an input feature for the toxicity head
3. **ChemBERTa-2 + LoRA (v2)**: single-seed for parity + 5-seed cluster ensemble, with concentration in the toxicity head and Tox21 aux head active
4. **Conformal-calibrated 95% prediction intervals**: empirical coverage measured on the held-out OOF folds
5. **Top-20 FDA IID candidates**: Pareto-ranked over (toxicity at 6 mol/kg, permeability, IRI), v2 filter applied, with the conformal PIs as uncertainty bars
6. **Figures for the README**: Spearman summary by task × split × architecture, 2D Pareto front with top-20 highlighted

Set the runtime to GPU before running (Runtime → Change runtime type → T4/A100/H100/Blackwell). The deep-env sanity-check cell will print versions and dry-run a LoRA adapter wrap before training, so any environment issue surfaces at install-verification time rather than 50 lines into ChemBERTa fine-tuning.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
![ -d cpa-screening ] || git clone https://github.com/cmendoza1031/cpa-screening.git
%cd /content/cpa-screening
!git pull --ff-only

In [ ]:
# Base deps (rdkit, pandas, scikit-learn, ...). NO --quiet so install errors are visible.
!pip install -r requirements.txt

In [ ]:
# Sanity check: fail loudly here rather than silently inside the pipeline.
import importlib
REQUIRED = ['rdkit', 'pubchempy', 'pandas', 'numpy', 'sklearn', 'scipy', 'matplotlib', 'requests', 'pyarrow']
missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
        print(f'  ok  {mod}')
    except ImportError as e:
        print(f'  MISSING  {mod}: {e}')
        missing.append(mod)
if missing:
    raise RuntimeError(f'Missing required modules: {missing}. Re-run the pip install cell, then retry.')
print('\nAll base deps available.')

## 2. Build the dataset

In [ ]:
# Build the consolidated long-format dataset (DOLMEN + Higgins Jan/Dec + Tox21
# + FDA IID candidate pool with the v2 filter).
#
# Tox21 in v2 downloads directly from the DeepChem GitHub mirror (no DeepChem
# Python package needed); the older --skip-tox21 escape hatch is still there
# if you want to skip the ~10s extra download. --skip-fda if you want to skip
# the ~5-min FDA candidate scoring pass during fast iteration.
#
# Audit summary lands in data/processed/audit.json regardless of skips.
!python -m src.data
# !python -m src.data --skip-fda

In [ ]:
import json, pathlib
audit = json.loads(pathlib.Path('data/processed/audit.json').read_text())
print(json.dumps(audit, indent=2)[:2000])

## 3. Random Forest baseline

Eval scheme: 70/15/15 for IRI (n=303), 5-fold CV for the small tasks (toxicity n=22, permeability n=16). LOO-CV permeability is reported as an RF-only sanity check (the gap between 5-fold and LOO is informative about scaffold leakage).

In [ ]:
!python -m src.train --model rf --seed 0

In [ ]:
import pandas as pd
rf_df = pd.read_csv('results/results_table.csv')
rf_df

## 4. ChemBERTa-2 + LoRA

Now install the deep-learning extras and run the multi-task ChemBERTa fine-tune. **Only run this section if you have a GPU runtime selected** (Runtime → Change runtime type → GPU). On CPU it will technically run but takes ~30+ min and the gradient noise on small batches makes results unstable.

Model: `DeepChem/ChemBERTa-77M-MLM` (RoBERTa-style, 384 hidden), mean-pool, 3 regression heads (toxicity, permeability, iri), LoRA (rank 8, alpha 16, dropout 0.1) on q/k/v.

**v2 changes here**: the toxicity head takes a scalar concentration alongside the pooled embedding, so the model can learn dose-response. The Tox21 auxiliary classification head is on by default (`--tox21-aux`); it adds a 12-class BCE loss on a shared 7,800-compound toxicity dataset, weighted at 0.1 so it regularizes the encoder without dominating the CPA gradient.

In [ ]:
# Install deep-learning extras. NOTE: no -U. -U would force-upgrade
# torch to 2.11+cu130 while torchvision stays at +cu128 (Colab ships them
# paired by CUDA version), and torchvision._check_cuda_version() then
# refuses to load -- which cascades into peft/transformers import failures.
# Without -U, pip leaves Colab's torch alone (it already satisfies >=2.1)
# and only upgrades the actual constraint-violator (torchao 0.10 -> 0.16+).
!pip install -r requirements-deep.txt

In [ ]:
# Hard sanity check on the deep-learning environment. If anything below is
# wrong we want to know NOW, not 50 lines into ChemBERTa training. Specifically
# we previously hit: pre-installed torchao 0.10 -> peft LoRA dispatcher
# raises ImportError when probing is_torchao_available().
import importlib.metadata as md
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected. Switch via Runtime -> Change runtime type.')

REQ_MIN = {'torch': '2.1', 'transformers': '4.40', 'peft': '0.10', 'accelerate': '0.27', 'torchao': '0.16'}
def _v(s):
    return tuple(int(p) for p in s.split('.')[:2] if p.isdigit())
bad = []
for pkg, minv in REQ_MIN.items():
    try:
        got = md.version(pkg)
        ok = _v(got) >= _v(minv)
        print(f"  {'ok ' if ok else 'BAD'}  {pkg:14s} {got} (need >= {minv})")
        if not ok:
            bad.append(f'{pkg} {got} < {minv}')
    except md.PackageNotFoundError:
        bad.append(f'{pkg} not installed')
        print(f'  BAD  {pkg:14s} NOT INSTALLED')
if bad:
    raise RuntimeError(f'Deep-learning deps need fixing: {bad}. Restart runtime + re-run requirements-deep.txt cell.')

# Probe the actual failure path: peft loading a LoRA adapter on RoBERTa.
from transformers import AutoModel
from peft import LoraConfig, get_peft_model
_m = AutoModel.from_pretrained('DeepChem/ChemBERTa-77M-MLM')
_cfg = LoraConfig(r=8, lora_alpha=16, target_modules=['query','key','value'], lora_dropout=0.1, bias='none', task_type='FEATURE_EXTRACTION')
_pm = get_peft_model(_m, _cfg)
_pm.print_trainable_parameters()
print('peft LoRA wrap OK')
del _m, _pm  # free memory before training

In [ ]:
# Single-seed rank-8 ChemBERTa+LoRA training, 5-fold CV across all compounds
# for direct apples-to-apples comparison with the RF baseline (which evaluates
# the small Higgins tasks via 5-fold CV). Trains 5 fresh models, each holding
# out 1/5 of compounds; OOF predictions are aggregated for per-task metrics.
#
# v2: --tox21-aux activates the auxiliary classification head on the shared
# encoder, with the BCE loss weighted at 0.1 (cpa_huber + 0.1 * tox21_bce).
# The toxicity head is concentration-aware in v2 even without the flag --
# concentration is part of the data layout, not a CLI option.
#
# Defaults: 30 epochs/fold, batch 32, lr 5e-5, early-stop patience 5,
# bf16 autocast on supported GPUs (A100/H100/Blackwell), fp32 fallback on T4.
!python -m src.train --model chemberta --lora-rank 8 --seed 0 --cv --cv-folds 5 --tox21-aux

In [ ]:
import pandas as pd
all_df = pd.read_csv('results/results_table.csv')
# Show side-by-side: RF + ChemBERTa
all_df.sort_values(['task', 'split', 'model']).reset_index(drop=True)

In [ ]:
from IPython.display import Image, display
import pathlib
# Show ChemBERTa parity plots (val + test for each task)
for p in sorted(pathlib.Path('results/figures').glob('parity_chemberta_*.png')):
    print(p.name)
    display(Image(str(p)))

## 5. Bundle results for download

In [ ]:
!zip -qr results.zip results data/processed/audit.json && ls -la results.zip

In [ ]:
from google.colab import files
files.download('results.zip')

---
## Cluster-aware split + 5-seed deep ensemble + conformal calibration

Three things stacked:
1. **Cluster-aware splits**: Butina clustering on Morgan FP r=2 / 2048-bit Tanimoto with threshold 0.6, then whole-cluster assignment to k-folds. This prevents test compounds from being scaffold-similar to anything in train. The whole 326-compound dataset collapses to 123 clusters (max=54, with DOLMEN sugars/amino acids forming one big cluster, and 83 singletons).
2. **5-seed deep ensembles** for both architectures, producing per-compound (mean, std) predictions; std is the epistemic-uncertainty proxy.
3. **Split-conformal calibration**: q95 = quantile of |y_true − ensemble_mean| at level (n+1)(1−α)/n, applied uniformly. Empirical coverage measured on the held-out OOF set as a sanity check (target 0.95).

Then `src.score_candidates` trains a separate full-data ensemble (no held-out), predicts on the ~1.8k FDA-IID candidates that pass the CPA-like filter, and Pareto-ranks the top-20.

In [ ]:
# RF cluster-split 5-seed ensemble (cheap; ~30s)
!python -m src.train --model rf --seed 0 --n-seeds 5 --split-mode cluster --cv-folds 5

In [ ]:
# ChemBERTa cluster-split 5-seed ensemble (5 seeds * 5 folds = 25 trainings,
# ~12-15 min on Blackwell). v2: --tox21-aux active.
!python -m src.train --model chemberta --lora-rank 8 --seed 0 --n-seeds 5 --split-mode cluster --cv --cv-folds 5 --tox21-aux

In [ ]:
# Show all results so far (RF and ChemBERTa, random and cluster, ensemble and single-seed)
import pandas as pd
df = pd.read_csv('results/results_table.csv')
df.sort_values(['task', 'scheme', 'model']).reset_index(drop=True)

### FDA IID candidate scoring + Pareto top-20

Trains a second ensemble (no held-out) on all CPA labels for each architecture, predicts on the FDA-IID candidate pool, applies the OOF-derived q95 from above for 95% prediction intervals, and Pareto-ranks. ChemBERTa is the primary architecture for the headline ranking; RF predictions are also saved.

Compute: ~5 min on Blackwell for ChemBERTa scoring (5 seeds, no CV) + RF nearly free.

In [ ]:
# Use --architecture both to get both RF and ChemBERTa predictions (ChemBERTa primary)
!python -m src.score_candidates --architecture both --n-seeds 5 --top-k 20 --tox21-aux

In [ ]:
import pandas as pd
top = pd.read_csv('results/candidates/top20.csv')
show_cols = ['ingredient_name', 'cas', 'pareto', 'composite_score',
             'toxicity_mean', 'toxicity_std',
             'permeability_mean', 'permeability_std',
             'iri_mean', 'iri_std']
top[show_cols]

In [ ]:
from IPython.display import Image, display
import pathlib
for p in sorted(pathlib.Path('results/figures').glob('pareto_3d_*.png')):
    print(p)
    display(Image(str(p)))

In [ ]:
# Generate the README figures from the cumulative results_table.csv +
# candidates/top20.csv (Spearman-summary bar chart + 2D Pareto with top-20).
!python -m src.figures

In [ ]:
from IPython.display import Image, display
import pathlib
for p in [
    pathlib.Path('results/figures/spearman_summary.png'),
    pathlib.Path('results/figures/pareto_2d_top20.png'),
    pathlib.Path('results/figures/pareto_3d_chemberta.png'),
]:
    if p.exists():
        print(p)
        display(Image(str(p)))

### 6. Mixture-aware analysis

Higgins Dec 2025 publishes binary mixture viability data only as bar charts in Figures 3-4. The readable values are 16 binary mixtures, all glycerol-paired, at 6 mol/kg total (3+3 each) or 12 mol/kg total (6+6 each). Sixteen rows is too few to fit a learned-interaction pair encoder (the architecture is documented in `src/models/mixture.py` for when more data becomes available). What we *can* do at this scale is quantify how badly an additive single-compound baseline fails on these 16 known mixtures, and rank the FDA candidate pool's binary pairs.

Two outputs from this cell:

1. **Additive-baseline evaluation** on the 16 known mixtures: per-rule (max / mean / sum_then_cap / weighted_max) Spearman / MAE / R² on viability prediction. The headline is the formamide+glycerol@12 mol/kg row, which is the famous neutralization case from Higgins's paper (actual viability 95, additive baseline predicts 0-40).

2. **FDA mixture pair scoring**: 140 v2 candidates choose 2 = 9,730 binary pairs at 6 mol/kg total (3+3 each), composite-scored over (toxicity, permeability, IRI) using the additive baseline. Pareto top-20 saved to `results/mixtures/top20_pairs.csv`.

In [ ]:
# Mixture analysis: additive baseline on 16 known mixtures + FDA pair scoring.
# Uses the v2 RF ensemble (concentration-aware toxicity, the strongest
# single-compound model in v2, cluster Spearman 0.64). ~3 min on CPU; FDA pair
# scoring dominates the runtime (~9700 pairs).
!python -m src.score_mixtures --architecture rf --n-seeds 5 --rule max --conc-total 6.0 --top-k 20

In [ ]:
import json, pathlib
import pandas as pd

mix_dir = pathlib.Path("results/mixtures")

# Additive baseline summary across the four combination rules
summary_path = mix_dir / "additive_baseline_summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print("Additive baseline on 16 known Higgins mixtures:")
    rows = []
    for rule, m in summary.items():
        rows.append({
            "rule": rule, "n": m["n"],
            "Spearman": round(m["spearman"], 3),
            "MAE": round(m["mae"], 2),
            "R^2": round(m["r2"], 3),
            "neutralization_misses": m["n_neutralization_misses"],
        })
    print(pd.DataFrame(rows).to_string(index=False))
    print()

# Per-row residuals on the formamide cases (the headline neutralization story)
pred_path = mix_dir / "additive_baseline_predictions.csv"
if pred_path.exists():
    pred = pd.read_csv(pred_path)
    formamide_rows = pred[
        pred["name_a"].str.contains("formamide", case=False, na=False)
        | pred["name_b"].str.contains("formamide", case=False, na=False)
    ]
    if not formamide_rows.empty:
        cols = ["name_a", "name_b", "conc_total", "viability_4c"]
        for r in ["max", "mean", "sum_then_cap", "weighted_max"]:
            if f"v_pred_{r}" in formamide_rows.columns:
                cols.append(f"v_pred_{r}")
        print("Formamide-containing mixtures (the neutralization story):")
        print(formamide_rows[cols].round(2).to_string(index=False))

# Top-20 FDA mixture pairs
top_path = mix_dir / "top20_pairs.csv"
if top_path.exists():
    top = pd.read_csv(top_path)
    print(f"\nTop-{len(top)} FDA mixture pairs (rule=max, total=6 mol/kg, additive baseline):")
    cols = ["rank", "name_a", "name_b", "tox_pred", "perm_pred", "iri_pred",
            "composite_score"]
    print(top[cols].round(3).to_string(index=False))

In [ ]:
# Bundle everything for download
!zip -qr results.zip results data/processed/audit.json && ls -la results.zip
from google.colab import files
files.download('results.zip')

---
## Done

Everything written by this run:

- `data/processed/audit.json`: dataset counts, label ranges, PubChem hit rate
- `results/results_table.csv`: cumulative metrics across every model × split × scheme
- `results/summary.json`: most recent run's full metrics + audit + per-fold breakdown
- `results/figures/`: parity plots per (model, task, split) + spearman summary + 2D Pareto top-20
- `results/candidates/top20.csv`: Pareto-ranked top-20 FDA IID compounds with calibrated 95% PIs
- `results/candidates/all_scored.csv`: full ranking for the ~435 CPA-filtered FDA candidates

The headline numbers, interpretation, top-20 chemistry read, and limitations are written up in the [README](https://github.com/cmendoza1031/cpa-screening). The mixture-aware extension, MD-ML coupling sketch, and active-learning loop are in [DESIGN_DOC.md](https://github.com/cmendoza1031/cpa-screening/blob/main/DESIGN_DOC.md).